In [1]:
import torch
from PIL import Image
import requests
from transformers import ViTImageProcessor, ViTModel # Example with ViT
import torch.nn.functional as F

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


In [3]:
# --- 1. Load ViT Model and Processor ---
# You can choose different models like "google/vit-base-patch16-224-in21k"
# or "facebook/deit-base-distilled-patch16-224"
model_name = "google/vit-base-patch16-224-in21k"
model = ViTModel.from_pretrained(model_name).to(device)
processor = ViTImageProcessor.from_pretrained(model_name)
model.eval() # Set to evaluation mode

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

C:\Users\bhunp\python312\Lib\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bhunp\.cache\huggingface\hub\models--google--vit-base-patch16-224-in21k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

ViTModel(
  (embeddings): ViTEmbeddings(
    (patch_embeddings): ViTPatchEmbeddings(
      (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): ViTEncoder(
    (layer): ModuleList(
      (0-11): 12 x ViTLayer(
        (attention): ViTSdpaAttention(
          (attention): ViTSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (output): ViTSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (intermediate): ViTIntermediate(
          (dense): Linear(in_features=768, out_features=3072, bias=True)
          (intermediate_act_fn): GELUAct

In [6]:
# --- 2. Load Your Images (same as CLIP example) ---
try:
    image1_pil = Image.open("check_cutting/skv.142801_ÐºÐµÑ€Ð½1-5_obj000_cls0.png").convert("RGB")
    image2_pil = Image.open("check_cutting/skv.142801_ÐºÐµÑ€Ð½1-5_obj003_cls0.png").convert("RGB")
    image3_pil = Image.open("check_cutting/skv.142801_ÐºÐµÑ€Ð½11-15_obj002_cls0.png").convert("RGB")
except Exception as e:
    print(f"Error loading images: {e}")

In [7]:
# --- 3. Preprocess Images and Get Embeddings ---
def get_vit_embedding(image_pil, model, processor, device):
    """Processes an image and returns its ViT embedding."""
    with torch.no_grad():
        inputs = processor(images=image_pil, return_tensors="pt").to(device)
        outputs = model(**inputs)
        # For ViT, we can use the pooler_output or the [CLS] token's representation
        # pooler_output is often a good choice for a general purpose embedding
        if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
            embedding = outputs.pooler_output.squeeze(0)
        else:
            # Fallback to CLS token (first token of last_hidden_state)
            embedding = outputs.last_hidden_state[:, 0, :].squeeze(0)
    return embedding

embedding1_vit = get_vit_embedding(image1_pil, model, processor, device)
embedding2_vit = get_vit_embedding(image2_pil, model, processor, device)
embedding3_vit = get_vit_embedding(image3_pil, model, processor, device)

print(f"ViT Embedding 1 shape: {embedding1_vit.shape}") # e.g., torch.Size([768]) for ViT-Base
print(f"ViT Embedding 2 shape: {embedding2_vit.shape}")
print(f"ViT Embedding 3 shape: {embedding3_vit.shape}")

# --- 4. Compare Embeddings (Cosine Similarity) ---
embedding1_vit_norm = F.normalize(embedding1_vit, p=2, dim=0)
embedding2_vit_norm = F.normalize(embedding2_vit, p=2, dim=0)
embedding3_vit_norm = F.normalize(embedding3_vit, p=2, dim=0)

similarity_1_2_vit = F.cosine_similarity(embedding1_vit_norm, embedding2_vit_norm, dim=0)
similarity_1_3_vit = F.cosine_similarity(embedding1_vit_norm, embedding3_vit_norm, dim=0)

print(f"\nViT Cosine Similarity (Image 1 vs Image 2): {similarity_1_2_vit.item():.4f}")
print(f"ViT Cosine Similarity (Image 1 vs Image 3): {similarity_1_3_vit.item():.4f}")

ViT Embedding 1 shape: torch.Size([768])
ViT Embedding 2 shape: torch.Size([768])
ViT Embedding 3 shape: torch.Size([768])

ViT Cosine Similarity (Image 1 vs Image 2): 0.5138
ViT Cosine Similarity (Image 1 vs Image 3): 0.3733
